<div style="display: flex; align-items: center;">

  <!-- Logos -->
  <div style="white-space: nowrap;">
    <img 
      src="https://www.upc.edu/comunicacio/ca/identitat/descarrega-arxius-grafics/fitxers-marca-principal/upc-positiu-p3005.png" 
      width="300"
      style="vertical-align: middle;"
    >
    <img 
      src="https://www.hipotecalowcost.com/wp-content/uploads/2019/08/Logo-CaixaBank.png" 
      width="200"
      style="vertical-align: middle;"
    >
  </div>

  <!-- Texto -->
  <div style="margin-left: auto; margin-right: 10px; text-align: right;">
      <p style="margin: 0;"><b>CaixaBank · Advanced Analytics Program</b></p>
      <p style="margin: 0;"><b>Model Risk & Data Science Training</b></p>
      <p style="margin: 0;">Automatización de procesos</p>
  </div>

</div>

# **RAG: documentos, chunking, embeddings y búsqueda semántica**
## *Construir respuestas sustentadas en una base documental*

En este laboratorio construiremos un sistema RAG desde cero.

La arquitectura será:

```text
documentos
   ↓
carga
   ↓
limpieza
   ↓
chunking
   ↓
representación vectorial
   ↓
búsqueda
   ↓
contexto recuperado
   ↓
respuesta
```

Primero utilizaremos una representación local con TF-IDF para entender el mecanismo sin depender de APIs externas.

Después veremos cómo sustituir esa capa por embeddings reales y un LLM.

## Objetivos

Al finalizar deberías poder:

1. Explicar qué es RAG.
2. Diferenciar recuperación y generación.
3. Cargar una pequeña base documental.
4. Dividir documentos en chunks.
5. Añadir metadatos.
6. Crear una representación vectorial.
7. Calcular similitud.
8. Recuperar top-k fragmentos.
9. Construir contexto.
10. Evitar responder cuando no hay evidencia suficiente.
11. Citar la procedencia del contexto.
12. Evaluar consultas.
13. Entender embeddings.
14. Diseñar una arquitectura RAG local o conectada a un LLM.

## 1. Preparación

In [ ]:
from pathlib import Path
import math
import re

import pandas as pd
import matplotlib.pyplot as plt

try:
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.metrics.pairwise import cosine_similarity
    SKLEARN_DISPONIBLE = True
except ImportError:
    SKLEARN_DISPONIBLE = False

RAIZ = Path("demo_rag")

for carpeta in [
    RAIZ / "documents",
    RAIZ / "outputs"
]:
    carpeta.mkdir(parents=True, exist_ok=True)

print("scikit-learn:", SKLEARN_DISPONIBLE)
print("Proyecto:", RAIZ.resolve())

## 2. ¿Qué es RAG?

RAG significa **Retrieval-Augmented Generation**.

La idea es sencilla:

```text
pregunta
   ↓
buscar información relevante
   ↓
recuperar fragmentos
   ↓
entregar contexto al modelo
   ↓
generar respuesta basada en ese contexto
```

La parte clave es separar:

- **retrieval**: encontrar evidencia;
- **generation**: redactar la respuesta.

## 3. Crear una base documental local

In [ ]:
documentos = {
    "politica_vacaciones.txt": '''
La empresa dispone de 24 días laborables de vacaciones al año.
Las vacaciones deben solicitarse con al menos 15 días de antelación.
La aprobación depende del responsable directo.
''',

    "politica_gastos.txt": '''
Los gastos inferiores a 100 euros pueden aprobarse por el responsable de equipo.
Los gastos entre 100 y 1000 euros requieren aprobación del responsable de departamento.
Los gastos superiores a 1000 euros requieren aprobación de dirección.
''',

    "teletrabajo.txt": '''
La modalidad de teletrabajo permite hasta tres días por semana.
Los equipos deben garantizar disponibilidad entre las 10:00 y las 14:00.
El material corporativo debe utilizarse conforme a la política de seguridad.
''',

    "soporte.txt": '''
Las incidencias críticas deben reportarse mediante el canal prioritario.
El tiempo objetivo de primera respuesta para incidencias críticas es de 30 minutos.
Las incidencias normales se atienden en horario laboral.
'''
}

for nombre, contenido in documentos.items():
    ruta = RAIZ / "documents" / nombre
    ruta.write_text(
        textwrap.dedent(contenido).strip() + "\n",
        encoding="utf-8"
    )

print(
    "Documentos creados:",
    len(documentos)
)

## 3.1 Visualizar documentos

In [ ]:
for ruta in sorted(
    (RAIZ / "documents").glob("*.txt")
):
    print("=" * 60)
    print(ruta.name)
    print(ruta.read_text(encoding="utf-8"))

## 4. Cargar documentos con metadatos

In [ ]:
def cargar_documentos(
    carpeta
):
    registros = []

    for ruta in sorted(
        Path(carpeta).glob("*.txt")
    ):
        registros.append({
            "documento":
                ruta.name,

            "ruta":
                str(ruta),

            "texto":
                ruta.read_text(
                    encoding="utf-8"
                ).strip()
        })

    return pd.DataFrame(registros)


df_docs = cargar_documentos(
    RAIZ / "documents"
)

df_docs

## 5. Chunking

Un documento largo suele dividirse en fragmentos.

Esto ayuda a:

- recuperar solo la parte relevante;
- reducir ruido;
- controlar el contexto;
- mejorar trazabilidad.

El tamaño óptimo depende del tipo de documento y del modelo.

## 5.1 Chunking por frases

In [ ]:
def dividir_en_frases(
    texto
):
    partes = re.split(
        r"(?<=[.!?])\s+",
        texto.strip()
    )

    return [
        parte.strip()
        for parte in partes
        if parte.strip()
    ]


ejemplo_frases = dividir_en_frases(
    df_docs.iloc[0]["texto"]
)

ejemplo_frases

## 5.2 Crear chunks con metadatos

In [ ]:
def crear_chunks(
    dataframe
):
    chunks = []

    chunk_id = 1

    for _, fila in dataframe.iterrows():

        frases = dividir_en_frases(
            fila["texto"]
        )

        for posicion, frase in enumerate(
            frases,
            start=1
        ):
            chunks.append({
                "chunk_id":
                    chunk_id,

                "documento":
                    fila["documento"],

                "posicion":
                    posicion,

                "texto":
                    frase
            })

            chunk_id += 1

    return pd.DataFrame(
        chunks
    )


df_chunks = crear_chunks(
    df_docs
)

df_chunks

## 5.3 Visualizar cantidad de chunks

In [ ]:
conteo_chunks = (
    df_chunks[
        "documento"
    ]
    .value_counts()
)

plt.figure(
    figsize=(8, 5)
)

plt.bar(
    conteo_chunks.index,
    conteo_chunks.values
)

plt.ylabel("Chunks")
plt.title("Chunks por documento")
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()

## 6. ¿Qué es un embedding?

Un embedding representa texto como un vector numérico.

Conceptualmente:

```text
"vacaciones anuales"
      ↓
[0.12, -0.41, 0.88, ...]
```

Textos semánticamente parecidos deberían quedar cerca en el espacio vectorial.

En este laboratorio utilizaremos TF-IDF como sustituto didáctico local para entender:

- vectorización;
- similitud;
- top-k.

## 7. Vectorización TF-IDF

In [ ]:
if SKLEARN_DISPONIBLE:

    vectorizador = TfidfVectorizer(
        lowercase=True,
        strip_accents="unicode",
        ngram_range=(1, 2)
    )

    matriz_chunks = vectorizador.fit_transform(
        df_chunks["texto"]
    )

    print(
        "Forma matriz:",
        matriz_chunks.shape
    )

## 7.1 Inspeccionar vocabulario

In [ ]:
if SKLEARN_DISPONIBLE:

    vocabulario = vectorizador.get_feature_names_out()

    print(
        "Términos:",
        len(vocabulario)
    )

    print(
        vocabulario[:40]
    )

## 8. Primera consulta

In [ ]:
consulta = (
    "¿Cuántos días de vacaciones tengo?"
)

print(consulta)

## 8.1 Vectorizar consulta

In [ ]:
if SKLEARN_DISPONIBLE:

    vector_consulta = vectorizador.transform(
        [consulta]
    )

    similitudes = cosine_similarity(
        vector_consulta,
        matriz_chunks
    )[0]

    similitudes

## 8.2 Añadir score a chunks

In [ ]:
if SKLEARN_DISPONIBLE:

    resultados = (
        df_chunks.copy()
    )

    resultados[
        "score"
    ] = similitudes

    resultados = resultados.sort_values(
        "score",
        ascending=False
    )

    resultados.head(5)

## 9. Función de búsqueda top-k

In [ ]:
def buscar_chunks(
    consulta,
    vectorizador,
    matriz,
    dataframe_chunks,
    top_k=3
):
    vector = vectorizador.transform(
        [consulta]
    )

    scores = cosine_similarity(
        vector,
        matriz
    )[0]

    resultados = (
        dataframe_chunks.copy()
    )

    resultados[
        "score"
    ] = scores

    return (
        resultados
        .sort_values(
            "score",
            ascending=False
        )
        .head(
            top_k
        )
        .reset_index(
            drop=True
        )
    )


if SKLEARN_DISPONIBLE:

    buscar_chunks(
        "¿Quién aprueba un gasto de 500 euros?",
        vectorizador,
        matriz_chunks,
        df_chunks,
        top_k=3
    )

## 10. Interpretar scores

La similitud no significa verdad.

Solo indica proximidad entre consulta y fragmento según la representación utilizada.

Por eso debemos definir un umbral mínimo de recuperación.

## 10.1 Visualizar scores

In [ ]:
if SKLEARN_DISPONIBLE:

    prueba = buscar_chunks(
        "¿Cuánto teletrabajo puedo hacer?",
        vectorizador,
        matriz_chunks,
        df_chunks,
        top_k=5
    )

    plt.figure(
        figsize=(8, 5)
    )

    plt.bar(
        prueba[
            "chunk_id"
        ].astype(str),
        prueba[
            "score"
        ]
    )

    plt.xlabel("Chunk")
    plt.ylabel("Similitud")
    plt.title("Top-k chunks recuperados")
    plt.tight_layout()
    plt.show()

## 11. Construir contexto

In [ ]:
def construir_contexto(
    resultados
):
    bloques = []

    for _, fila in resultados.iterrows():

        bloques.append(
            f"[Fuente: {fila['documento']} | "
            f"chunk {fila['chunk_id']}]\n"
            f"{fila['texto']}"
        )

    return "\n\n".join(
        bloques
    )


if SKLEARN_DISPONIBLE:

    recuperados = buscar_chunks(
        "¿Quién aprueba un gasto superior a 1000 euros?",
        vectorizador,
        matriz_chunks,
        df_chunks,
        top_k=3
    )

    contexto = construir_contexto(
        recuperados
    )

    print(contexto)

## 12. Construir prompt RAG

In [ ]:
if SKLEARN_DISPONIBLE:

    pregunta = (
        "¿Quién aprueba un gasto superior a 1000 euros?"
    )

    prompt_rag = f'''
Responde exclusivamente con la información del contexto.

Si la respuesta no está en el contexto, responde:
"No dispongo de información suficiente."

Incluye la fuente utilizada.

CONTEXTO:
{contexto}

PREGUNTA:
{pregunta}
'''

    print(
        prompt_rag
    )

### Interpretación

El prompt RAG debe controlar:

- fuentes;
- alcance;
- qué hacer cuando no hay evidencia;
- trazabilidad.

## 13. Respuesta simulada basada en contexto

In [ ]:
respuesta_rag_simulada = {
    "respuesta":
        "Los gastos superiores a 1000 euros requieren aprobación de dirección.",

    "fuente":
        "politica_gastos.txt"
}

respuesta_rag_simulada

## 14. Consulta sin evidencia

In [ ]:
if SKLEARN_DISPONIBLE:

    fuera_de_base = buscar_chunks(
        "¿Cuál es el salario del director general?",
        vectorizador,
        matriz_chunks,
        df_chunks,
        top_k=3
    )

    fuera_de_base

## 14.1 Aplicar umbral de evidencia

In [ ]:
def decidir_si_hay_evidencia(
    resultados,
    umbral=0.15
):
    if len(
        resultados
    ) == 0:
        return False

    return (
        resultados.iloc[0][
            "score"
        ]
        >= umbral
    )


if SKLEARN_DISPONIBLE:

    print(
        decidir_si_hay_evidencia(
            fuera_de_base,
            umbral=0.15
        )
    )

### Interpretación

Si la recuperación es débil, es preferible no inventar una respuesta.

## 15. Pipeline RAG local completo

In [ ]:
def rag_local(
    pregunta,
    vectorizador,
    matriz,
    chunks,
    top_k=3,
    umbral=0.15
):
    recuperados = buscar_chunks(
        pregunta,
        vectorizador,
        matriz,
        chunks,
        top_k=top_k
    )

    hay_evidencia = (
        decidir_si_hay_evidencia(
            recuperados,
            umbral=umbral
        )
    )

    if not hay_evidencia:
        return {
            "pregunta":
                pregunta,

            "estado":
                "sin_evidencia",

            "contexto":
                "",

            "fuentes":
                [],

            "recuperados":
                recuperados
        }

    contexto = construir_contexto(
        recuperados
    )

    fuentes = (
        recuperados[
            "documento"
        ]
        .drop_duplicates()
        .tolist()
    )

    return {
        "pregunta":
            pregunta,

        "estado":
            "evidencia_encontrada",

        "contexto":
            contexto,

        "fuentes":
            fuentes,

        "recuperados":
            recuperados
    }


if SKLEARN_DISPONIBLE:

    resultado_rag = rag_local(
        "¿Cuánto tiempo tarda soporte en responder una incidencia crítica?",
        vectorizador,
        matriz_chunks,
        df_chunks
    )

    print(
        resultado_rag[
            "estado"
        ]
    )

    print(
        resultado_rag[
            "contexto"
        ]
    )

## 16. Evaluar varias preguntas

In [ ]:
preguntas = [
    "¿Cuántos días de vacaciones hay?",
    "¿Quién aprueba un gasto de 500 euros?",
    "¿Cuántos días se puede teletrabajar?",
    "¿Cuál es el tiempo objetivo para incidencia crítica?",
    "¿Cuál es el salario del CEO?"
]

if SKLEARN_DISPONIBLE:

    evaluacion = []

    for pregunta in preguntas:

        resultado = rag_local(
            pregunta,
            vectorizador,
            matriz_chunks,
            df_chunks
        )

        max_score = (
            float(
                resultado[
                    "recuperados"
                ][
                    "score"
                ].max()
            )
            if len(
                resultado[
                    "recuperados"
                ]
            )
            else 0
        )

        evaluacion.append({
            "pregunta":
                pregunta,

            "estado":
                resultado[
                    "estado"
                ],

            "max_score":
                max_score,

            "fuentes":
                ", ".join(
                    resultado[
                        "fuentes"
                    ]
                )
        })

    df_evaluacion = pd.DataFrame(
        evaluacion
    )

    df_evaluacion

## 16.1 Visualizar scores por consulta

In [ ]:
if SKLEARN_DISPONIBLE:

    plt.figure(
        figsize=(10, 5)
    )

    plt.bar(
        range(
            len(
                df_evaluacion
            )
        ),
        df_evaluacion[
            "max_score"
        ]
    )

    plt.xticks(
        range(
            len(
                df_evaluacion
            )
        ),
        [
            f"Q{i+1}"
            for i in range(
                len(
                    df_evaluacion
                )
            )
        ]
    )

    plt.ylabel("Máxima similitud")
    plt.title("Confianza de recuperación por consulta")
    plt.tight_layout()
    plt.show()

## 17. Chunk demasiado grande vs demasiado pequeño

In [ ]:
comparativa_chunking = pd.DataFrame({
    "tamano": [
        "Muy pequeño",
        "Intermedio",
        "Muy grande"
    ],
    "ventaja": [
        "Alta precisión local",
        "Equilibrio",
        "Mucho contexto"
    ],
    "riesgo": [
        "Pierde contexto",
        "Debe ajustarse",
        "Introduce ruido"
    ]
})

comparativa_chunking

## 18. Overlap

En documentos largos podemos solapar fragmentos:

```text
chunk 1: palabras 1-100
chunk 2: palabras 81-180
chunk 3: palabras 161-260
```

El solapamiento reduce el riesgo de cortar información importante justo en el límite.

## 18.1 Chunking por palabras con overlap

In [ ]:
def chunk_palabras(
    texto,
    tamano=20,
    overlap=5
):
    palabras = texto.split()

    chunks = []

    inicio = 0

    while inicio < len(
        palabras
    ):

        fin = inicio + tamano

        chunks.append(
            " ".join(
                palabras[
                    inicio:fin
                ]
            )
        )

        if fin >= len(
            palabras
        ):
            break

        inicio += (
            tamano
            - overlap
        )

    return chunks


chunk_palabras(
    df_docs.iloc[0][
        "texto"
    ],
    tamano=10,
    overlap=3
)

## 19. Metadatos

Los chunks deberían conservar información como:

- documento;
- página;
- sección;
- fecha;
- tipo;
- permisos;
- identificador.

Esto permite:

- citar;
- filtrar;
- aplicar seguridad;
- depurar.

## 19.1 Ejemplo de metadatos

In [ ]:
metadata_demo = {
    "documento":
        "politica_gastos.pdf",

    "pagina":
        4,

    "seccion":
        "Aprobaciones",

    "departamento":
        "Finanzas",

    "confidencial":
        False
}

metadata_demo

## 20. Filtrado antes de búsqueda

In [ ]:
chunks_con_area = df_chunks.copy()

mapa_area = {
    "politica_vacaciones.txt":
        "RRHH",

    "politica_gastos.txt":
        "Finanzas",

    "teletrabajo.txt":
        "RRHH",

    "soporte.txt":
        "IT"
}

chunks_con_area[
    "area"
] = chunks_con_area[
    "documento"
].map(
    mapa_area
)

chunks_con_area.head()

## 20.1 Filtrar por área

In [ ]:
chunks_rrhh = chunks_con_area[
    chunks_con_area[
        "area"
    ] == "RRHH"
]

chunks_rrhh

### Interpretación

En RAG real es frecuente filtrar primero por metadatos y después hacer búsqueda vectorial.

## 21. Arquitectura con embeddings reales

Sustitución conceptual:

```text
TF-IDF
   ↓
Embedding model
```

Ejemplos de componentes:

- modelo de embeddings;
- base vectorial;
- retriever;
- LLM.

Arquitectura:

```text
documentos
   ↓
chunking
   ↓
embeddings
   ↓
vector store
   ↓
retriever
   ↓
LLM
```

## 22. Vector store: concepto

In [ ]:
vector_stores = pd.DataFrame({
    "tipo": [
        "En memoria",
        "FAISS",
        "Chroma",
        "pgvector",
        "Vector DB gestionada"
    ],
    "uso": [
        "POC",
        "Local",
        "Local/app",
        "PostgreSQL",
        "Producción escalable"
    ]
})

vector_stores

## 23. RAG vs fine-tuning

In [ ]:
comparativa = pd.DataFrame({
    "criterio": [
        "Actualizar conocimiento",
        "Citar fuentes",
        "Documentos privados",
        "Cambiar comportamiento",
        "Coste de actualización"
    ],
    "RAG": [
        "Fácil",
        "Sí",
        "Adecuado",
        "Limitado",
        "Bajo"
    ],
    "Fine-tuning": [
        "Reentrenar",
        "No por defecto",
        "No es su función principal",
        "Sí",
        "Mayor"
    ]
})

comparativa

## 24. Errores frecuentes de RAG

In [ ]:
errores_rag = pd.DataFrame({
    "problema": [
        "Chunking pobre",
        "Embedding inadecuado",
        "Top-k incorrecto",
        "Sin umbral",
        "Contexto irrelevante",
        "Prompt permisivo",
        "Sin citas"
    ],
    "consecuencia": [
        "Recuperación débil",
        "Similitud pobre",
        "Ruido o falta de contexto",
        "Respuestas sin evidencia",
        "Alucinación",
        "Modelo inventa",
        "Sin trazabilidad"
    ]
})

errores_rag

## 25. Evaluación RAG

Un sistema RAG no se evalúa solo preguntando “¿suena bien?”.

Debemos analizar al menos:

1. ¿recuperó el fragmento correcto?;
2. ¿la respuesta usa ese fragmento?;
3. ¿evita inventar?;
4. ¿cita la fuente correcta?;
5. ¿sabe abstenerse?;
6. ¿qué latencia tiene?;
7. ¿qué coste tiene?

## 25.1 Dataset de evaluación

In [ ]:
evaluacion_rag = pd.DataFrame({
    "pregunta": [
        "Vacaciones anuales",
        "Gasto de 500 euros",
        "Teletrabajo",
        "Incidencia crítica"
    ],
    "documento_esperado": [
        "politica_vacaciones.txt",
        "politica_gastos.txt",
        "teletrabajo.txt",
        "soporte.txt"
    ]
})

evaluacion_rag

## 25.2 Evaluar recuperación

In [ ]:
if SKLEARN_DISPONIBLE:

    resultados_eval = []

    preguntas_eval = [
        "¿Cuántos días de vacaciones tengo?",
        "¿Quién aprueba un gasto de 500 euros?",
        "¿Cuántos días puedo teletrabajar?",
        "¿Cuánto tarda una incidencia crítica?"
    ]

    for pregunta, esperado in zip(
        preguntas_eval,
        evaluacion_rag[
            "documento_esperado"
        ]
    ):
        recuperados = buscar_chunks(
            pregunta,
            vectorizador,
            matriz_chunks,
            df_chunks,
            top_k=1
        )

        obtenido = (
            recuperados.iloc[0][
                "documento"
            ]
        )

        resultados_eval.append({
            "pregunta":
                pregunta,

            "esperado":
                esperado,

            "obtenido":
                obtenido,

            "correcto":
                esperado == obtenido
        })

    df_eval_final = pd.DataFrame(
        resultados_eval
    )

    df_eval_final

## 25.3 Accuracy de retrieval

In [ ]:
if SKLEARN_DISPONIBLE:

    accuracy = (
        df_eval_final[
            "correcto"
        ].mean()
    )

    print(
        f"Retrieval accuracy: {accuracy:.1%}"
    )

## 26. Integración con n8n

El flujo podría ser:

```text
Webhook
   ↓
pregunta
   ↓
Retriever / Vector Store
   ↓
chunks
   ↓
LLM
   ↓
respuesta con fuente
   ↓
Respond to Webhook
```

El siguiente proyecto final combinará:

- n8n;
- RAG;
- agente;
- herramientas.

## 27. Guardar chunks

In [ ]:
ruta_chunks = (
    RAIZ
    / "outputs"
    / "chunks.csv"
)

df_chunks.to_csv(
    ruta_chunks,
    index=False
)

print(
    ruta_chunks
)

## 28. Mini-práctica integrada

Construye un RAG local con al menos:

- 10 documentos;
- 30 chunks;
- metadatos;
- vectorización;
- top-k;
- umbral;
- contexto;
- fuentes;
- 10 preguntas de evaluación.

El pipeline debe:

1. cargar;
2. limpiar;
3. chunkear;
4. vectorizar;
5. buscar;
6. filtrar;
7. construir contexto;
8. abstenerse si no hay evidencia;
9. devolver fuentes;
10. medir retrieval accuracy.

## 29. Ejercicios finales

### Ejercicio 1
Crea cinco documentos.

### Ejercicio 2
Cárgalos en DataFrame.

### Ejercicio 3
Chunking por frases.

### Ejercicio 4
Añade chunk_id.

### Ejercicio 5
Vectoriza.

### Ejercicio 6
Haz consulta.

### Ejercicio 7
Calcula similitud.

### Ejercicio 8
Top-k.

### Ejercicio 9
Construye contexto.

### Ejercicio 10
Añade fuentes.

### Ejercicio 11
Define umbral.

### Ejercicio 12
Prueba una pregunta fuera de base.

### Ejercicio 13
Implementa abstención.

### Ejercicio 14
Chunking con overlap.

### Ejercicio 15
Añade metadatos.

### Ejercicio 16
Filtra por área.

### Ejercicio 17
Evalúa retrieval.

### Ejercicio 18
Reto: construye un RAG local completo y prepáralo para sustituir TF-IDF por embeddings reales.

## 30. Preguntas de reflexión

1. ¿Qué significa RAG?
2. ¿Qué diferencia hay entre retrieval y generation?
3. ¿Por qué chunkear?
4. ¿Qué es un embedding?
5. ¿Qué representa similitud coseno?
6. ¿Qué significa top-k?
7. ¿Por qué definir umbral?
8. ¿Por qué citar fuentes?
9. ¿Qué ocurre si no hay evidencia?
10. ¿Qué aporta metadata filtering?
11. ¿Qué diferencia hay entre RAG y fine-tuning?
12. ¿Qué evaluarías en un RAG?
13. ¿Qué cambiaría al sustituir TF-IDF por embeddings?
14. ¿Cómo conectarías este sistema con n8n?

## 31. Cierre

Ya tenemos:

```text
documentos
   ↓
chunks
   ↓
vectores
   ↓
retrieval
   ↓
contexto
   ↓
respuesta sustentada
```

El siguiente notebook integrará todo el curso en el proyecto final:

```text
Smart Office Assistant
   ↓
n8n
   ↓
RAG
   ↓
agente
   ↓
herramientas
   ↓
informes / acciones
```